# FCC Invoices Dataset Import to S3 Vector store

This notebook demonstrates how to import the FCC invoices (REALKIE) dataset into S3 Vectors for use with the dynamic few-shot Lambda function.

The FCC invoices dataset contains invoice documents that can be used as few-shot examples for document extraction tasks.

## Process Overview:

1. **Load FCC Invoices Dataset** - Sync and load the dataset using load_dataset()
2. **Generate Embeddings** - Create multimodal embeddings using Amazon Nova
3. **Upload to S3 Vectors** - Store embeddings and metadata in S3 Vectors index
4. **Verify Import** - Test similarity search functionality

> **Note**: This notebook requires AWS credentials with permissions for Bedrock, S3, and S3 Vectors services.

## 1. Install Dependencies

In [ ]:
# Let's make sure that modules are autoreloaded
%load_ext autoreload
%autoreload 2

ROOTDIR="../../../"
# First uninstall existing package (to ensure we get the latest version)
%pip uninstall -y idp_common

# Install the IDP common package with all components in development mode
%pip install -q -e "{ROOTDIR}/lib/idp_common_pkg[dev, all]"

# Note: We can also install specific components like:
# %pip install -q -e "{ROOTDIR}/lib/idp_common_pkg[ocr,classification,extraction,evaluation]"

# Check installed version
%pip show idp_common | grep -E "Version|Location"

# Install required packages
%pip install -q pillow tqdm pandas datasets matplotlib

# Optionally use a .env file fxor environment variables
try:
    from dotenv import load_dotenv
    load_dotenv()  
except ImportError:
    pass

## 2. Import Libraries

In [ ]:
import json
import subprocess
from pathlib import Path
from typing import Dict, List, Any
from tqdm import tqdm
import pandas as pd
import io

import boto3
from datasets import load_dataset

# Import IDP common modules
from idp_common import bedrock

print("Libraries imported successfully")

## 3. Configure S3 Vectors and Bedrock

In [ ]:
# Configuration - Update these values from the IDP stack in CloudFormation Resources tab
GENAIIDP_S3_WORKING_BUCKET = "<s3-working-bucket>" # From IDP stack Resources tab

S3_VECTORS_BUCKET = "genaiidp-dynamic-few-shot"
S3_VECTORS_INDEX = "documents"
EMBEDDING_MODEL_ID = "amazon.nova-2-multimodal-embeddings-v1:0"
EMBEDDING_DIMENSIONS = 3072

# Initialize clients
s3vectors_client = boto3.client('s3vectors')
s3_client = boto3.client('s3')
bedrock_client = bedrock.BedrockClient()

print(f"Configured for dataset S3 Bucket: {GENAIIDP_S3_WORKING_BUCKET}")
print(f"Configured for S3 Vectors bucket: {S3_VECTORS_BUCKET}")
print(f"Configured for S3 Vectors index: {S3_VECTORS_INDEX}")
print(f"Using embedding model: {EMBEDDING_MODEL_ID}")
print(f"Using embedding dimensions: {EMBEDDING_DIMENSIONS}")

## 4. Load FCC Invoices Dataset

In [ ]:
# Sync FCC invoices dataset from S3
print("Syncing FCC invoices dataset from S3...")

# Configuration for this dataset
CLASS_LABEL = 'Invoice'

# Create datasets directory
dataset_root_dir = Path('../datasets')
dataset_root_dir.mkdir(exist_ok=True)

# Dataset directory
dataset_dir = dataset_root_dir / 'fcc_invoices'

# Sync dataset from S3 using AWS CLI with Wasabi endpoint
if not dataset_dir.exists() or not any(dataset_dir.iterdir()):
    print("Syncing dataset from S3...")
    sync_command = [
        'aws', 's3', 'sync',
        's3://project-fruitfly/fcc_invoices',
        str(dataset_dir),
        '--endpoint-url=https://s3.us-east-2.wasabisys.com',
        '--no-sign-request'
    ]
    
    try:
        result = subprocess.run(sync_command, capture_output=True, text=True, check=True)
        print(f"Dataset synced successfully to {dataset_dir}")
        print(f"Sync output: {result.stdout}")
    except subprocess.CalledProcessError as e:
        print(f"Error syncing dataset: {e}")
        print(f"Error output: {e.stderr}")
        raise
else:
    print(f"Using existing dataset at {dataset_dir}")

# Load the training dataset using load_dataset
print("Loading training dataset...")
try:
    # Load dataset from local directory
    dataset = load_dataset('csv', data_dir=str(dataset_dir), split='train')
    print(f"Loaded dataset with {len(dataset)} samples")
    
    # Show sample information
    if len(dataset) > 0:
        sample = dataset[0]
        print(f"Sample keys: {list(sample.keys())}")
        if 'image' in sample:
            print(f"Sample image size: {sample['image'].size}")
        
except Exception as e:
    print(f"Error loading dataset: {e}")
    # Fallback: list files in directory
    image_files = list(dataset_dir.glob('**/*.jpg')) + list(dataset_dir.glob('**/*.png'))
    print(f"Found {len(image_files)} image files in directory")
    if image_files:
        print(f"Sample image: {image_files[0].name}")
        print(f"Image file size: {image_files[0].stat().st_size} bytes")

print(f"Class label: {CLASS_LABEL}")

## 5. Process Dataset and Generate Embeddings

In [ ]:
def upload_image_to_s3(image_bytes: bytes, s3_key: str) -> str:
    """Upload image to S3 and return S3 URI."""
    s3_client.put_object(
        Bucket=GENAIIDP_S3_WORKING_BUCKET,
        Key=s3_key,
        Body=image_bytes,
        ContentType='image/jpeg'
    )
    return f"s3://{GENAIIDP_S3_WORKING_BUCKET}/{s3_key}"

def load_csv_labels():
    """Load the CSV file with labels and metadata."""
    csv_path = dataset_dir / 'train.csv'
    if csv_path.exists():
        try:
            df = pd.read_csv(csv_path)
            print(f"Loaded CSV with {len(df)} rows")
            return df
        except Exception as e:
            print(f"Error loading CSV: {e}")
            return None
    else:
        print(f"CSV file not found at {csv_path}")
        return None

def match_image_to_csv_row(image_path: str, csv_df: pd.DataFrame):
    """Match an image path to the corresponding CSV row."""
    if csv_df is None:
        return None
    
    # Extract the image filename from the path
    image_name = Path(image_path).name
    
    # Look for matching rows in the CSV
    for idx, row in csv_df.iterrows():
        image_files_str = row.get('image_files', '')
        if image_name in image_files_str:
            return row
    
    return None

def get_image_bytes_from_file(image_path):
    """Read image file directly as bytes."""
    with open(image_path, 'rb') as f:
        return f.read()

def create_sample_attributes_prompt() -> str:
    """Create a sample attributes prompt for FCC invoices based on the actual schema."""
    # Updated to match the actual FCC invoices dataset structure and expected JSON schema
    attributes_prompt = """expected attributes are:
        "Agency": "Great American Media",
        "Advertiser": "ISS/HOUSE MAJ PAC", 
        "GrossTotal": 94700.00,
        "PaymentTerms": "Cash In Advance",
        "AgencyCommission": 14205.00,
        "NetAmountDue": 80495.00,
        "LineItems": [
            {
                "LineItemDescription": "TODAY IN FLORIDA @9PM",
                "LineItemStartDate": "10/18/2016", 
                "LineItemEndDate": null,
                "LineItemDays": ["T"],
                "LineItemRate": 500.00
            },
            {
                "LineItemDescription": "CH 7 NEWS @ 10PM",
                "LineItemStartDate": "10/18/2016",
                "LineItemEndDate": null, 
                "LineItemDays": ["T"],
                "LineItemRate": 3200.00
            }
        ]
    """.strip()
    return attributes_prompt

def parse_ground_truth_labels(labels_json_str: str) -> Dict:
    """Parse ground truth labels from the dataset and convert to expected format."""
    import json
    
    try:
        labels = json.loads(labels_json_str)
    except (json.JSONDecodeError, TypeError):
        return None
    
    # Initialize the result structure
    result = {
        "Agency": None,
        "Advertiser": None,
        "GrossTotal": None,
        "PaymentTerms": None,
        "AgencyCommission": None,
        "NetAmountDue": None,
        "LineItems": []
    }
    
    # Group line items by their properties
    line_items = {}
    
    for label in labels:
        label_type = label.get('label', '')
        text = label.get('text', '')
        
        # Map top-level fields
        if label_type == 'Agency':
            result['Agency'] = text
        elif label_type == 'Advertiser':
            result['Advertiser'] = text
        elif label_type == 'Gross Total':
            try:
                result['GrossTotal'] = float(text.replace(',', '').replace('$', ''))
            except ValueError:
                result['GrossTotal'] = text
        elif label_type == 'Net Amount Due':
            try:
                result['NetAmountDue'] = float(text.replace(',', '').replace('$', ''))
            except ValueError:
                result['NetAmountDue'] = text
        elif label_type == 'Payment Terms':
            result['PaymentTerms'] = text
        elif label_type == 'Agency Commission':
            try:
                result['AgencyCommission'] = float(text.replace(',', '').replace('$', ''))
            except ValueError:
                result['AgencyCommission'] = text
        
        # Handle line items (group by position or create separate items)
        elif label_type.startswith('Line Item - '):
            field_name = label_type.replace('Line Item - ', '')
            start_pos = label.get('start', 0)
            
            # Use start position to group related line item fields
            # Find the closest line item group
            closest_key = None
            min_distance = float('inf')
            
            for key in line_items.keys():
                distance = abs(start_pos - key)
                if distance < min_distance and distance < 1000:  # Within reasonable range
                    min_distance = distance
                    closest_key = key
            
            if closest_key is None:
                closest_key = start_pos
                line_items[closest_key] = {}
            
            # Map field names to expected schema
            if field_name == 'Description':
                line_items[closest_key]['LineItemDescription'] = text
            elif field_name == 'Start Date':
                line_items[closest_key]['LineItemStartDate'] = text
            elif field_name == 'End Date':
                line_items[closest_key]['LineItemEndDate'] = text if text else None
            elif field_name == 'Rate':
                try:
                    line_items[closest_key]['LineItemRate'] = float(text.replace(',', '').replace('$', ''))
                except ValueError:
                    line_items[closest_key]['LineItemRate'] = text
            elif field_name == 'Days':
                # Convert day codes to day names
                day_mapping = {
                    'M': 'M', 'T': 'T', 'W': 'W', 'Th': 'Th', 'F': 'F', 'S': 'S', 'Su': 'Su',
                    '1': 'M', '2': 'T', '3': 'W', '4': 'Th', '5': 'F', '6': 'S', '7': 'Su'
                }
                days = []
                for char in text:
                    if char in day_mapping and char != '-':
                        mapped_day = day_mapping[char]
                        if mapped_day not in days:
                            days.append(mapped_day)
                line_items[closest_key]['LineItemDays'] = days
    
    # Convert line items dict to list
    result['LineItems'] = list(line_items.values())
    
    return result

def create_metadata(s3_image_uri: str, sample_data: Dict = None) -> Dict:
    """Create metadata for S3 Vectors entry."""
    class_prompt = f"This is an example of the class '{CLASS_LABEL}'"
    
    # If we have actual sample data with labels, use it to create a more accurate attributes prompt
    if sample_data and 'labels' in sample_data:
        parsed_labels = parse_ground_truth_labels(sample_data['labels'])
        if parsed_labels:
            attributes_prompt = f"expected attributes are: {json.dumps(parsed_labels, indent=2)}"
        else:
            attributes_prompt = create_sample_attributes_prompt()
    else:
        attributes_prompt = create_sample_attributes_prompt()

    return {
        "classLabel": CLASS_LABEL,
        "classPrompt": class_prompt,
        "attributesPrompt": attributes_prompt,
        "imagePath": s3_image_uri,
    }

print("Helper functions defined")

## 6. Import Dataset to S3 Vectors

In [ ]:
# Process a subset of the dataset (adjust as needed)
MAX_SAMPLES = 250  # Adjust this number based on your needs
BATCH_SIZE = 15    # Adjust this number based on your needs

# Load the CSV labels (this contains the image_files information)
csv_df = load_csv_labels()
if csv_df is None:
    print("Failed to load CSV data. Exiting.")
    raise Exception("CSV loading failed")

samples_to_process = min(MAX_SAMPLES, len(csv_df))
print(f"Processing {samples_to_process} samples from FCC invoices CSV data...")

vectors_to_upload = []
failed_samples = []

for i in tqdm(range(samples_to_process), desc="Processing samples"):
    try:
        csv_row = csv_df.iloc[i]
        
        # Get image files from the CSV row
        image_files_str = csv_row.get('image_files', '')
        if not image_files_str:
            print(f"No image files found for sample {i}")
            failed_samples.append(i)
            continue
            
        # Parse the image files array (it's stored as a JSON string)
        import json
        try:
            image_files = json.loads(image_files_str)
        except json.JSONDecodeError:
            print(f"Failed to parse image_files for sample {i}: {image_files_str}")
            failed_samples.append(i)
            continue
        
        # Use the first image file (or you could process all images)
        if not image_files:
            print(f"Empty image_files array for sample {i}")
            failed_samples.append(i)
            continue
            
        # Load the first image file
        image_file_path = image_files[0]
        full_image_path = dataset_root_dir / image_file_path
        
        if not full_image_path.exists():
            print(f"Image file not found: {full_image_path}")
            failed_samples.append(i)
            continue
            
        # Load image file as bytes
        image_bytes = get_image_bytes_from_file(full_image_path)

        # Upload image to S3
        s3_key = f"fcc_invoices/sample_{i:06d}.jpg"
        s3_image_uri = upload_image_to_s3(image_bytes, s3_key)
        
        # Generate embedding
        embedding = bedrock_client.generate_embedding(
            image_source=image_bytes,
            model_id=EMBEDDING_MODEL_ID,
            dimensions=EMBEDDING_DIMENSIONS
        )
        
        # Create metadata using the CSV row data
        sample_data = {'labels': csv_row.get('labels')}
        metadata = create_metadata(s3_image_uri, sample_data)

        # Prepare vector for upload
        vector_entry = {
            "key": f"fcc_invoices_sample_{i:06d}",
            "data": {"float32": embedding},
            "metadata": metadata
        }

        vectors_to_upload.append(vector_entry)
        
        # Upload in batches to avoid memory issues
        if len(vectors_to_upload) >= BATCH_SIZE:
            print(f"\nUploading batch of {len(vectors_to_upload)} vectors...")
            response = s3vectors_client.put_vectors(
                vectorBucketName=S3_VECTORS_BUCKET,
                indexName=S3_VECTORS_INDEX,
                vectors=vectors_to_upload
            )
            print(f"Batch upload response: {response.get('ResponseMetadata', {}).get('HTTPStatusCode')}")
            vectors_to_upload = []  # Clear batch
            
    except Exception as e:
        print(f"\nFailed to process sample {i}: {e}")
        failed_samples.append(i)
        continue

# Upload remaining vectors
if vectors_to_upload:
    print(f"\nUploading final batch of {len(vectors_to_upload)} vectors...")
    response = s3vectors_client.put_vectors(
        vectorBucketName=S3_VECTORS_BUCKET,
        indexName=S3_VECTORS_INDEX,
        vectors=vectors_to_upload
    )
    print(f"Final batch upload response: {response.get('ResponseMetadata', {}).get('HTTPStatusCode')}")

print(f"\nImport completed!")
print(f"Successfully processed: {samples_to_process - len(failed_samples)} samples from CSV data")
print(f"Failed samples: {len(failed_samples)}")
if failed_samples:
    print(f"Failed sample indices: {failed_samples[:10]}...")  # Show first 10

## 7. Verify Import with Similarity Search

In [ ]:
# Load test split for similarity search verification
test_dataset = load_dataset('csv', data_dir=str(dataset_dir), split='test')
print(f"Loaded test dataset with {len(test_dataset)} samples")

if test_dataset is not None and len(test_dataset) > 0:
    # Use the first sample from test split
    test_sample_index = 0
    test_csv_row = test_dataset[test_sample_index]
    
    # Get test image from CSV row
    test_image_files_str = test_csv_row.get('image_files', '')
    if test_image_files_str:
        try:
            test_image_files = json.loads(test_image_files_str)
            if test_image_files:
                test_image_path = dataset_root_dir / test_image_files[0]
                if test_image_path.exists():
                    test_image_bytes = get_image_bytes_from_file(test_image_path)
                    print(f"Loaded test image: {test_image_files[0]}")
                else:
                    print(f"Test image file not found: {test_image_path}")
                    test_image_bytes = None
            else:
                print("Empty image_files array in test sample")
                test_image_bytes = None
        except (json.JSONDecodeError, IndexError) as e:
            print(f"Failed to parse test image_files: {e}")
            test_image_bytes = None
    else:
        print("No image_files found in test sample")
        test_image_bytes = None
else:
    print("Test split is empty or could not be loaded")
    test_image_bytes = None

if test_image_bytes is not None:
    print(f"\nTesting similarity search with test sample {test_sample_index}...")

    # Generate embedding for test image
    test_embedding = bedrock_client.generate_embedding(
        image_source=test_image_bytes,
        model_id=EMBEDDING_MODEL_ID,
        dimensions=EMBEDDING_DIMENSIONS
    )
else:
    print("No test image available for similarity search verification.")
    test_embedding = None

if test_embedding is not None:
    # Query S3 Vectors for similar examples
    response = s3vectors_client.query_vectors(
        vectorBucketName=S3_VECTORS_BUCKET,
        indexName=S3_VECTORS_INDEX,
        queryVector={"float32": test_embedding},
        topK=5,
        returnDistance=True,
        returnMetadata=True
    )

    print(f"\nFound {len(response['vectors'])} similar examples:")
    for i, vector in enumerate(response['vectors']):
        distance = vector.get('distance', 'N/A')
        key = vector.get('key', 'N/A')
        metadata = vector.get('metadata', {})
        class_label = metadata.get('classLabel', 'N/A')
        class_prompt = metadata.get('classPrompt', 'N/A')
        attributes_prompt = metadata.get('attributesPrompt', 'N/A')
        image_path = metadata.get('imagePath', 'N/A')
        
        print(f"  {i+1}. Key: {key}")
        print(f"     Distance: {distance:.4f}")
        print(f"     Class Label: {class_label}")
        print(f"     Class Prompt: {class_prompt}")
        print(f"     Attributes Prompt: {attributes_prompt[:100]}...")  # Truncate for readability
        print(f"     Image Path: {image_path}")
        print()
else:
    print("Skipping similarity search - no test embedding available.")

# Display source image and found similar images
if test_image_bytes is not None and 'response' in locals() and response.get('vectors'):
    import matplotlib.pyplot as plt
    from PIL import Image as PILImage
    import io
    
    # Calculate number of images to display (source + top similar images)
    num_similar = min(3, len(response['vectors']))  # Show top 3 similar images
    total_images = 1 + num_similar  # Source + similar images
    
    # Create subplot layout
    fig, axes = plt.subplots(1, total_images, figsize=(5 * total_images, 6))
    if total_images == 1:
        axes = [axes]  # Make it iterable for single image
    
    # Display source image
    source_img = PILImage.open(io.BytesIO(test_image_bytes))
    axes[0].imshow(source_img)
    axes[0].set_title(f'Source Image (Test Sample {test_sample_index})', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Display similar images
    for i, vector in enumerate(response['vectors'][:num_similar]):
        try:
            # Get image path from metadata
            metadata = vector.get('metadata', {})
            image_s3_path = metadata.get('imagePath', '')
            distance = vector.get('distance', 0)
            
            if image_s3_path:
                # Extract S3 key from the full S3 URI
                s3_key = image_s3_path.replace(f's3://{GENAIIDP_S3_WORKING_BUCKET}/', '')
                
                # Download image from S3
                try:
                    response_obj = s3_client.get_object(Bucket=GENAIIDP_S3_WORKING_BUCKET, Key=s3_key)
                    image_data = response_obj['Body'].read()
                    similar_img = PILImage.open(io.BytesIO(image_data))
                    
                    # Display the image
                    axes[i + 1].imshow(similar_img)
                    axes[i + 1].set_title(f'Similar #{i+1}\nDistance: {distance:.3f}', fontsize=10)
                    axes[i + 1].axis('off')
                    
                except Exception as e:
                    # If can't load from S3, show placeholder
                    axes[i + 1].text(0.5, 0.5, f'Image not available\n{str(e)[:50]}...', 
                                    ha='center', va='center', transform=axes[i + 1].transAxes)
                    axes[i + 1].set_title(f'Similar #{i+1}\nDistance: {distance:.3f}', fontsize=10)
                    axes[i + 1].axis('off')
            else:
                # No image path available
                axes[i + 1].text(0.5, 0.5, 'No image path', ha='center', va='center', 
                                transform=axes[i + 1].transAxes)
                axes[i + 1].set_title(f'Similar #{i+1}\nDistance: {distance:.3f}', fontsize=10)
                axes[i + 1].axis('off')
                
        except Exception as e:
            print(f'Error displaying similar image {i+1}: {e}')
            axes[i + 1].text(0.5, 0.5, f'Error: {str(e)[:30]}...', ha='center', va='center', 
                            transform=axes[i + 1].transAxes)
            axes[i + 1].set_title(f'Similar #{i+1}', fontsize=10)
            axes[i + 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f'\nDisplayed source image and top {num_similar} similar images from the vector store.')
    
else:
    print('No images to display - either no test image was loaded or no similar images were found.')
    if test_image_bytes is None:
        print('Reason: No test image available')
    elif 'response' not in locals():
        print('Reason: No similarity search was performed')
    elif not response.get('vectors'):
        print('Reason: No similar images found in vector store')

## 8. Summary and Next Steps

In [ ]:
print("=== Few-shot Dataset Import Summary ===")
print(f"✅ Dataset: FCC Invoices (REALKIE)")
print(f"✅ Samples processed: {samples_to_process - len(failed_samples) if 'samples_to_process' in locals() and 'failed_samples' in locals() else 'N/A'}")
print(f"✅ S3 Vectors Bucket: {S3_VECTORS_BUCKET}")
print(f"✅ S3 Vectors Index: {S3_VECTORS_INDEX}")
print(f"✅ Images stored in: s3://{GENAIIDP_S3_WORKING_BUCKET}/fcc_invoices/")
print(f"✅ Embedding Model: {EMBEDDING_MODEL_ID}")
print(f"✅ Similarity search verified")

print("\n=== Next Steps ===")
print("1. ✅ Updated attributes mapping to match actual FCC invoices dataset structure")
print("2. ✅ Added ground truth label parsing from CSV data")
print("3. Configure your IDP extraction to use the dynamic few-shot Lambda ARN")
print("4. Test document processing with few-shot examples!")
print("5. Fine-tune the label parsing logic if needed based on your specific use case")